# Gold fact -- `dbo.fct_sales`

Sales at order-line grain

**Grain:** One row per product per order. An order with three distinct products produces three rows. This is the finest grain available and every other sales measure aggregates up from it.

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_silver, writes to wh_gold. Both must be
# attached to this notebook; wh_gold must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "wh_gold"
source_item = "lh_silver"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="fct_sales",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="gold", table_name="fct_sales")
print(f"load_id={load_id}  environment={environment}  table=fct_sales")

from ttfabric.warehouse import gold_target

gold = gold_target(
    spark,
    warehouse="wh_gold",
    schema="dbo",
    write_mode="warehouse_connector",
)


In [ ]:
# ---- Read silver -------------------------------------------------
df = spark.read.table(f"{source_item}.stg_order_items")


In [ ]:
# Join stg_orders (inner)
# Inner join is load-bearing: an order line whose header was quarantined must not reach gold.
stg_orders = spark.read.table(f"{source_item}.stg_orders")
df = df.join(stg_orders, on="order_id", how="inner")


In [ ]:
# Resolve customer_sk from dim_customer
# Point-in-time lookup on order_date: a fact joins the dimension
# version that was current when the event happened, not the
# version current today.
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_customer"),
    surrogate_key="customer_sk",
    lookup_on="customer_id",
    dimension_key="customer_id",
    dimension_surrogate_key="customer_sk",
    as_of_column="order_date",
)


In [ ]:
# Resolve product_sk from dim_product
# Point-in-time lookup on order_date: a fact joins the dimension
# version that was current when the event happened, not the
# version current today.
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_product"),
    surrogate_key="product_sk",
    lookup_on="product_id",
    dimension_key="product_id",
    dimension_surrogate_key="product_sk",
    as_of_column="order_date",
)


In [ ]:
# Resolve order_date_sk from dim_date
from ttfabric.dimensions import lookup_surrogate_key

df = lookup_surrogate_key(
    df,
    dimension=gold.read("dim_date"),
    surrogate_key="order_date_sk",
    lookup_on="order_date_key",
    dimension_key="full_date",
    dimension_surrogate_key="date_sk",
)


In [ ]:
# Business rule: revenue_recognition
# Revenue counts only once an order is confirmed. Pending orders may never complete and cancelled ones never did. The rule lives here so every report inherits one definition of revenue.
df = df.withColumn("recognised_revenue", F.expr("""CASE WHEN is_revenue THEN subtotal ELSE 0 END"""))


In [ ]:
# Business rule: line_cost
# PROVISIONAL -- placeholder logic, not real business data.
# Replace before this measure informs a decision.
# No cost column exists upstream. A flat 62% cost ratio stands in so margin is demonstrable. REPLACE WITH REAL COST DATA before this measure is put in front of anyone making a pricing decision.
df = df.withColumn("line_cost", F.expr("""round(subtotal * 0.62, 2)"""))


In [ ]:
# Business rule: gross_margin
# PROVISIONAL -- placeholder logic, not real business data.
# Replace before this measure informs a decision.
# 
df = df.withColumn("gross_margin", F.expr("""recognised_revenue - CASE WHEN is_revenue THEN line_cost ELSE 0 END"""))


In [ ]:
# ---- Rename to target names --------------------------------------
df = (df
    .withColumnRenamed("subtotal", "line_revenue")
)


In [ ]:
# ---- Write -------------------------------------------------------
final_columns = ['order_id', 'order_item_id', 'customer_sk', 'product_sk', 'order_date_sk', 'quantity', 'unit_price', 'line_revenue', 'recognised_revenue', 'line_cost', 'gross_margin', 'subtotal_was_corrected']
out = (df.select(*[c for c in final_columns if c in df.columns])
    .withColumn("_built_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id)))

gold.write(out, "fct_sales")
print(f"wrote {out.count():,} rows to fct_sales")


In [ ]:
# ---- Tests -------------------------------------------------------
#   GOLD-GRAIN-001: The declared grain holds -- no duplicate lines.
#   GOLD-FK-001: Every dimension key resolves, unknown member included.
#   GOLD-RECON-001: Gold revenue ties back to silver.
from ttfabric.quality import (assert_unique, assert_not_null,
                         assert_keys_resolve, assert_reconciles)

assert_unique(out, ['order_item_id'])
assert_not_null(out, ['customer_sk', 'product_sk', 'order_date_sk'])

# not_null is not enough: a failed lookup yields the unknown-member
# key, not a null, so a fact table with every key unresolved passes
# a not-null check while reporting everything against "Unknown".
assert_keys_resolve(out, ['customer_sk', 'product_sk', 'order_date_sk'])

# Gold must tie back to silver. Compared against the rows that
# actually reached gold -- the inner join legitimately drops
# lines whose header was quarantined, so comparing against all
# of silver would fail for a correct build.
reachable = (spark.read.table(f"{source_item}.stg_order_items")
    .join(spark.read.table(f"{source_item}.stg_orders")
          .select("order_id"),
          on="order_id", how="inner"))
expected = reachable.agg(F.sum("subtotal")).collect()[0][0] or 0
actual = out.agg(F.sum("line_revenue")).collect()[0][0] or 0
assert_reconciles(float(actual), float(expected), 0.01,
                  "fct_sales.line_revenue vs stg_order_items.subtotal")

dq.record_input(out.count())
dq.record_output(out.count())
dq.flush()
